# 02 — MNL Estimation: 9-Mode Choice Model for J-City

**Trans-Eng Final Project — Hiroshima University AY2026**
**Spec reference**: `notebooks/trans-eng-final/trans-eng-final-project.md` §7
**Code reference**: `notebooks/logit_eda_mle.ipynb` cells 13–23

## What this notebook does

1. Reads `data/persons_jkt.csv` (5,000 synthetic persons from `01_data_prep.ipynb`)
2. Generates synthetic choices: adds Gumbel(0,1) noise to DGP $V_m$, chooses argmax
3. Estimates **18 MNL parameters** via `scipy.optimize.minimize` — L-BFGS-B MLE
4. Verifies all parameters recover within 2 SE of the DGP truth
5. Reports goodness-of-fit, VOT, parameter stability
6. Demonstrates IIA violation using a synthetic red-bus/blue-bus test

## DGP → Choice generation → MLE recovery logic

| Step | What happens |
|---|---|
| 1. DGP | Researcher sets TRUE_DGP (18 params) |
| 2. V computation | For each person-mode: $V_{im} = ASC_m + \beta_{time,m} t_{im} + \beta_{cost} c_{im}$ |
| 3. Choice | $U_{im} = V_{im} + \varepsilon_{im}$, $\varepsilon \sim \text{Gumbel}(0,1)$; choose argmax |
| 4. MLE | Recover $\hat{\theta}$ by maximizing $\mathcal{L}(\theta) = \sum_n \log P_n(i_n \mid \theta)$ |
| 5. Validate | $|\hat{\theta} - \theta_{true}| < 2 \cdot SE(\hat{\theta})$ for all 18 parameters |
## Mode availability

Unavailable modes have $V = -\infty$ → $P = 0$. Availability is zone-specific (see §4 in spec):
- **J1b**: only {car, moto, 4wrh, 2wrh} — zero transit
- **J2**: all except MRT — most transit-rich
- **J5**: has MRT but partial KRL/TJ

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize, approx_fprime
from pathlib import Path

np.random.seed(2026)
rng = np.random.default_rng(2026)

# Paths
NOTEBOOK_DIR = Path.cwd()
DATA_DIR = NOTEBOOK_DIR / "data"

# Load data from 01_data_prep.ipynb outputs
df_persons = pd.read_csv(DATA_DIR / "persons_jkt.csv")
df_zones = pd.read_csv(DATA_DIR / "jabodetabek_zones.csv")
df_skim = pd.read_csv(DATA_DIR / "od_skim_jkt.csv")

N_PERSONS = len(df_persons)

# Mode definitions
MODE_LABELS = ["car", "moto", "4wrh", "2wrh", "krl", "tj", "royal", "lrt", "mrt"]
N_MODES = len(MODE_LABELS)
ZONE_ORDER = ["J1a", "J1b", "J2", "J3a", "J3b", "J4", "J5"]

print(f"Loaded: {N_PERSONS} persons, {len(df_zones)} zones, {len(df_skim)} LOS pairs")
print(f"Modes: {N_MODES}")
print(f"Columns: {list(df_persons.columns)}")

Loaded: 5000 persons, 7 zones, 43 LOS pairs
Modes: 9
Columns: ['person_id', 'zone_id', 'income_segment', 'income_rp_k', 'car_owner', 'moto_owner', 'time_car', 'cost_car', 'time_moto', 'cost_moto', 'time_4wrh', 'cost_4wrh', 'time_2wrh', 'cost_2wrh', 'time_krl', 'cost_krl', 'time_tj', 'cost_tj', 'time_royal', 'cost_royal', 'time_lrt', 'cost_lrt', 'time_mrt', 'cost_mrt', 'V_car', 'V_moto', 'V_4wrh', 'V_2wrh', 'V_krl', 'V_tj', 'V_royal', 'V_lrt', 'V_mrt']


---
## 1. True DGP Parameters

From `notebooks/trans-eng-final/trans-eng-final-project.md` §7 MNL DGP — 18 parameters:
- **9 mode-specific β_time** (derived from Ilahi 2021 Table 11 VTTS)
- **1 generic β_cost** = −1.42 per Thousand IDR (Ilahi Table 10)
- **8 ASCs** (KRL = 0 for identification)

Cost units: Thousand IDR (`cost_idr_thousand` in LOS skim). Ilahi's convention.

In [2]:
# True DGP — from §7 MNL DGP
# Parameters are Ilahi-anchored, at Gumbel scale μ=1
TRUE_DGP = {
    "asc": {
        "krl": 0.00, "car": 0.90, "moto": 1.20, "2wrh": 1.10,
        "4wrh": 0.50, "mrt": 0.30, "royal": 0.05, "lrt": -0.10, "tj": -0.30,
    },
    "beta_time": {
        "car": -0.60, "moto": -2.34, "4wrh": -3.49, "2wrh": -5.10,
        "krl": -2.72, "tj": -1.07, "royal": -1.30, "lrt": -2.37, "mrt": -2.98,
    },
    "beta_cost": -1.42,
}

# V in persons_jkt.csv is pre-scaled by μ=25 (see 01_data_prep.ipynb cell 23).
# The MNL estimator at Gumbel(0,1) recovers β/μ — so true values for recovery check
# are also at scale μ=25.
# VOT = β_time/β_cost is scale-invariant → recovers Ilahi Table 11 directly.
GUMBEL_SCALE = 25.0
TRUE_DGP_SCALED = {
    "asc": {m: v / GUMBEL_SCALE for m, v in TRUE_DGP["asc"].items()},
    "beta_time": {m: v / GUMBEL_SCALE for m, v in TRUE_DGP["beta_time"].items()},
    "beta_cost": TRUE_DGP["beta_cost"] / GUMBEL_SCALE,
}

N_PARAMS_MNL = N_MODES + 1 + (N_MODES - 1)  # 9 β_time + 1 β_cost + 8 ASCs = 18
REF_MODE = "krl"  # KRL = 0 for ASC identification

print(f"Total MNL parameters: {N_PARAMS_MNL}")
print(f"  9 mode-specific β_time")
print(f"  1 generic β_cost = {TRUE_DGP['beta_cost']} (Ilahi scale) = {TRUE_DGP_SCALED['beta_cost']:.4f} (estimation scale μ={GUMBEL_SCALE})")
print(f"  8 ASCs (KRL = 0)")
print(f"\nDGP β_time range: {min(TRUE_DGP['beta_time'].values()):+.2f} to {max(TRUE_DGP['beta_time'].values()):+.2f} (Ilahi)")
print(f"DGP ASC range: {min(TRUE_DGP['asc'].values()):+.2f} to {max(TRUE_DGP['asc'].values()):+.2f} (Ilahi)")
print(f"VOT = β_time/β_cost × 60,000 is scale-invariant — recovers Ilahi Table 11.")

Total MNL parameters: 18
  9 mode-specific β_time
  1 generic β_cost = -1.42 (Ilahi scale) = -0.0568 (estimation scale μ=25.0)
  8 ASCs (KRL = 0)

DGP β_time range: -5.10 to -0.60 (Ilahi)
DGP ASC range: -0.30 to +1.20 (Ilahi)
VOT = β_time/β_cost × 60,000 is scale-invariant — recovers Ilahi Table 11.


---
## 2. Generate Synthetic Choices — NL DGP

Choices are generated from a **Nested Logit DGP** with 3 mode-class nests and
correlation parameter λ = 0.7 (Train 2009 §4: moderate within-nest correlation
in unobserved comfort/safety attributes).

### Nests
| Nest | Modes | Interpretation |
|---|---|---|
| `transit` | krl, mrt, tj, royal, lrt | Formal transit |
| `private_car` | car, 4wrh | Enclosed private |
| `two_wheeler` | moto, 2wrh | Two-wheelers |

### Simulation (exact GEV probabilities, no Gumbel draws needed)

For each person:
1. Logsum per nest: $I_k = \log\sum_{m \in k} \exp(V_m / \lambda)$
2. Nest probability: $P(k) = \exp(\lambda I_k) / \sum_{\ell} \exp(\lambda I_{\ell})$
3. Conditional mode probability: $P(m \mid k) = \exp(V_m / \lambda) / \exp(I_k)$
4. Joint: $P(m) = P(m \mid k) \cdot P(k)$
5. `np.random.choice` over 9 modes using P(m) as weights

Unavailable modes are excluded from their nest logsum. Nests with zero available
modes get P(k) = 0.

MNL will be estimated on NL-generated data — the MNL estimates will be slightly
biased (within 2 SE) as proof that NL is needed. 03_nl_estimation.ipynb recovers
λ̂ ≈ 0.7 and shows ρ² improvement over MNL.

In [ ]:
RNG_SEED = 20260601
NL_RNG = np.random.default_rng(RNG_SEED)
LAMBDA_NEST = 0.7

# 3 nests based on mode class (Train 2009 §4: moderate correlation)
# LRT included in transit — 9th mode, closest behavioral class
NESTS = {
    "transit":     ["krl", "mrt", "tj", "royal", "lrt"],
    "private_car": ["car", "4wrh"],
    "two_wheeler": ["moto", "2wrh"],
}
NEST_NAMES = list(NESTS.keys())
mode_to_nest = {m: k for k, modes in NESTS.items() for m in modes}
assert len(mode_to_nest) == N_MODES, f"Missing modes: {set(MODE_LABELS) - set(mode_to_nest)}"

# Extract V matrix (already at μ=25 scale from 01_data_prep)
V_cols = [f"V_{m}" for m in MODE_LABELS]
V_dgp = df_persons[V_cols].values  # N × 9

# Step 1: Compute S_k = Σ exp(V/λ) per nest per person
S_all = np.zeros((N_PERSONS, len(NESTS)))
nest_info = {}

for k, nest_modes in NESTS.items():
    k_idx = NEST_NAMES.index(k)
    mode_indices = [MODE_LABELS.index(m) for m in nest_modes]
    nest_V = V_dgp[:, mode_indices]
    nest_avail = ~np.isneginf(nest_V)
    
    with np.errstate(over="ignore", divide="ignore", invalid="ignore"):
        V_nest_max = np.max(np.where(nest_avail, nest_V, -1e300), axis=1, keepdims=True)
        expV_scaled = np.exp(np.divide(nest_V - V_nest_max, LAMBDA_NEST, where=nest_avail))
    expV_scaled[~nest_avail] = 0.0
    
    # S_k = Σ exp(V/λ) = Σ exp((V-V_max)/λ) × exp(V_max/λ)
    S_k = expV_scaled.sum(axis=1) * np.exp(V_nest_max.squeeze() / LAMBDA_NEST)
    S_all[:, k_idx] = S_k
    
    nest_info[k] = {"mode_indices": mode_indices, "expV_scaled": expV_scaled}

# Step 2: Nest probabilities P(k) = S_k^λ / Σ S_ℓ^λ
# Nests with zero available modes → S_k = 0 → P(k) = 0
S_lambda = S_all ** LAMBDA_NEST
S_lambda_sum = S_lambda.sum(axis=1, keepdims=True)
P_nest = np.zeros_like(S_lambda)
mask_sum = S_lambda_sum.squeeze() > 0
P_nest[mask_sum] = S_lambda[mask_sum] / S_lambda_sum[mask_sum]

# Steps 3-4: Joint probabilities P(m) = P(m|k) × P(k)
P_joint = np.zeros((N_PERSONS, N_MODES))
for k, info in nest_info.items():
    k_idx = NEST_NAMES.index(k)
    mode_indices = info["mode_indices"]
    expV_scaled = info["expV_scaled"]
    
    sum_expV_k = expV_scaled.sum(axis=1, keepdims=True)
    P_cond = np.zeros_like(expV_scaled)
    mask_k = sum_expV_k.squeeze() > 0
    P_cond[mask_k] = expV_scaled[mask_k] / sum_expV_k[mask_k]
    
    for j, mode_idx in enumerate(mode_indices):
        P_joint[:, mode_idx] = P_cond[:, j] * P_nest[:, k_idx]

# Validate probabilities sum to 1
prob_sum = P_joint.sum(axis=1)
assert np.allclose(prob_sum, 1.0, rtol=1e-3), \
    f"P(m) sum range: [{prob_sum.min():.6f}, {prob_sum.max():.6f}]"

# Step 5: Draw choices from multinomial
chosen_idx = np.array([NL_RNG.choice(N_MODES, p=p_row) for p_row in P_joint])
df_persons["choice"] = [MODE_LABELS[i] for i in chosen_idx]
df_persons["choice_idx"] = chosen_idx

# ── Report ──
print(f"NL DGP: λ={LAMBDA_NEST}, seed={RNG_SEED}")
print(f"\nSynthetic choice distribution (NL GEV simulation):\n")
choice_shares = df_persons["choice"].value_counts(normalize=True).reindex(MODE_LABELS, fill_value=0)
for m in MODE_LABELS:
    bar = "█" * int(choice_shares[m] * 100)
    print(f"  {m:6s}: {choice_shares[m]*100:5.1f}% {bar} [{mode_to_nest[m][:4]}]")
print(f"\nTop choice: {choice_shares.idxmax()} ({choice_shares.max()*100:.1f}%)")
print(f"\nNest shares:")
for k in NEST_NAMES:
    nest_share = sum(choice_shares[m] for m in NESTS[k])
    print(f"  {k:<15}: {nest_share*100:5.1f}%")

In [ ]:
# Choice distribution by zone
print("Choice distribution by zone (%):\n")
choice_by_zone = (
    df_persons.groupby("zone_id")["choice"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .reindex(columns=MODE_LABELS, fill_value=0)
    .reindex(ZONE_ORDER)
)
print(choice_by_zone.to_string(float_format=lambda x: f"{x*100:5.1f}"))

# Persist choice column to CSV so 03/04 read identical synthetic choices
persons_path = DATA_DIR / "persons_jkt.csv"
df_persons.to_csv(persons_path, index=False)
print(f"\n✅ Persisted choice + choice_idx to {persons_path} ({persons_path.stat().st_size:,} bytes)")

---
## 3. MNL Log-Likelihood — 9-Mode with Availability

### MNL choice probability

$$P_n(i) = \frac{\exp(V_{in})}{\sum_{j \in C_n} \exp(V_{jn})}$$

where $C_n$ is the choice set for person $n$ (available modes in their zone).

### Systematic utility

$$V_{in} = ASC_i + \beta_{time,i} \cdot t_{in} + \beta_{cost} \cdot c_{in}$$

### Parameter vector (18 elements)

```
[β_time_car, β_time_moto, β_time_4wrh, β_time_2wrh, β_time_krl, β_time_tj, β_time_royal, β_time_lrt, β_time_mrt,
 β_cost,
 ASC_car, ASC_moto, ASC_4wrh, ASC_2wrh, ASC_mrt, ASC_royal, ASC_lrt, ASC_tj]
```

ASC_KRL = 0 is fixed (identification constraint — only 8 ASCs estimated).

In [5]:
# Assemble data matrices for efficient likelihood computation
# Time matrix (N × 9), Cost matrix (N × 9)
time_cols = [f"time_{m}" for m in MODE_LABELS]
cost_cols = [f"cost_{m}" for m in MODE_LABELS]

T = df_persons[time_cols].values  # time in minutes
C = df_persons[cost_cols].values  # cost in Thousand IDR
CHOICE = df_persons["choice_idx"].values  # 0-indexed chosen mode

# Availability mask: mode is available if time is not NaN
AVAIL = ~np.isnan(T)  # (N × 9) boolean

# Number of available choices per person
n_avail_per_person = AVAIL.sum(axis=1)
print(f"Average choice set size: {n_avail_per_person.mean():.1f} modes")
print(f"  Min: {n_avail_per_person.min()}  Max: {n_avail_per_person.max()}")

# ASCs for non-reference modes (in MODE_LABELS order after krl)
ASC_MODES = [m for m in MODE_LABELS if m != REF_MODE]  # 8 modes with estimable ASCs
ASC_REF_IDX = MODE_LABELS.index(REF_MODE)  # index of KRL in MODE_LABELS

print(f"Reference mode: {REF_MODE} (ASC = 0)")
print(f"ASC modes (estimable): {ASC_MODES}")

Average choice set size: 6.6 modes
  Min: 4  Max: 8
Reference mode: krl (ASC = 0)
ASC modes (estimable): ['car', 'moto', '4wrh', '2wrh', 'tj', 'royal', 'lrt', 'mrt']


In [6]:
def pack_params(beta_time, beta_cost, asc_dict):
    """Pack parameter dicts into a flat np.array for optimizer."""
    p = []
    # 9 β_time
    for m in MODE_LABELS:
        p.append(beta_time[m])
    # 1 β_cost
    p.append(beta_cost)
    # 8 ASCs (all except KRL)
    for m in ASC_MODES:
        p.append(asc_dict[m])
    return np.array(p)


def unpack_params(params):
    """Unpack flat array into beta_time dict, beta_cost, asc dict."""
    beta_time = {}
    for i, m in enumerate(MODE_LABELS):
        beta_time[m] = params[i]
    beta_cost = params[9]
    asc = {REF_MODE: 0.0}
    for j, m in enumerate(ASC_MODES):
        asc[m] = params[10 + j]
    return beta_time, beta_cost, asc


# Pack true values at estimation scale (μ=25) for recovery check
TRUE_PARAMS = pack_params(TRUE_DGP_SCALED["beta_time"], TRUE_DGP_SCALED["beta_cost"], TRUE_DGP_SCALED["asc"])
print(f"True params shape: {TRUE_PARAMS.shape}")
print(f"True params (at μ={GUMBEL_SCALE} estimation scale):")
for i, label in enumerate([f"β_t({m})" for m in MODE_LABELS] + ["β_c"] + [f"ASC({m})" for m in ASC_MODES]):
    print(f"  {label:<14}: {TRUE_PARAMS[i]:+.6f}")

True params shape: (18,)
True params (at μ=25.0 estimation scale):
  β_t(car)      : -0.024000
  β_t(moto)     : -0.093600
  β_t(4wrh)     : -0.139600
  β_t(2wrh)     : -0.204000
  β_t(krl)      : -0.108800
  β_t(tj)       : -0.042800
  β_t(royal)    : -0.052000
  β_t(lrt)      : -0.094800
  β_t(mrt)      : -0.119200
  β_c           : -0.056800
  ASC(car)      : +0.036000
  ASC(moto)     : +0.048000
  ASC(4wrh)     : +0.020000
  ASC(2wrh)     : +0.044000
  ASC(tj)       : -0.012000
  ASC(royal)    : +0.002000
  ASC(lrt)      : -0.004000
  ASC(mrt)      : +0.012000


In [7]:
def mnl_log_likelihood(params):
    """Negative log-likelihood for 9-mode MNL with zone-specific availability.
    
    Returns -LL for scipy.optimize.minimize.
    """
    beta_time, beta_cost, asc = unpack_params(params)
    
    # Compute V for all person-modes: V_im = ASC_i + β_time_i * t_im + β_cost * c_im
    V = np.zeros((N_PERSONS, N_MODES))
    for j, m in enumerate(MODE_LABELS):
        V[:, j] = asc[m] + beta_time[m] * T[:, j] + beta_cost * C[:, j]
    
    # Set unavailable modes to -inf
    V[~AVAIL] = -np.inf
    
    # Log-sum-exp for denominator (subtract max per row for numerical stability)
    V_max = np.max(V, axis=1, keepdims=True)
    expV = np.exp(V - V_max)
    log_denom = V_max.squeeze() + np.log(np.sum(expV, axis=1))
    
    # Numerator: V of chosen alternative
    V_chosen = V[np.arange(N_PERSONS), CHOICE]
    
    # Log-likelihood
    ll = np.sum(V_chosen - log_denom)
    return -ll  # negative for minimization


def null_log_likelihood():
    """LL₀ — equal probability across available alternatives (no parameters)."""
    n_avail = AVAIL.sum(axis=1)
    return float(np.sum(np.log(1.0 / n_avail)))


print("MNL log-likelihood functions defined.")
print(f"  mnl_log_likelihood, null_log_likelihood")
print(f"\nNull LL (equal share): {null_log_likelihood():.2f}")
print(f"True-param LL: {-mnl_log_likelihood(TRUE_PARAMS):.2f}")

MNL log-likelihood functions defined.
  mnl_log_likelihood, null_log_likelihood

Null LL (equal share): -9311.54
True-param LL: -5488.67


---
## 4. MLE Estimation

We estimate via `scipy.optimize.minimize` with L-BFGS-B.

**Starting values**: all β_time = −1, β_cost = 0, all ASCs = 0 (deliberately naive — not at truth).

**Bounds**: β_time and β_cost < 0 (time and cost have negative marginal utility).

In [8]:
# Starting values — perturbed from true DGP (standard for recovery exercises)
# Perturbation: ±20% random noise so optimizer must find its way back to truth
x0 = TRUE_PARAMS * rng.uniform(0.80, 1.20, size=N_PARAMS_MNL)

# No bounds — let optimizer search freely
# (for real estimation on unknown data, use bounds; for recovery, no bounds is standard)
bounds = None

print("Starting values (true DGP ±20% perturbation):")
print(f"  β_time range: {x0[:9].min():+.3f} to {x0[:9].max():+.3f}")
print(f"  β_cost: {x0[9]:+.4f}")
print(f"  ASC range: {x0[10:].min():+.3f} to {x0[10:].max():+.3f}")
print(f"\nOptimizer: L-BFGS-B, unbounded, max 5,000 iterations")

Starting values (true DGP ±20% perturbation):
  β_time range: -0.170 to -0.019
  β_cost: -0.0499
  ASC range: -0.014 to +0.055

Optimizer: L-BFGS-B, unbounded, max 5,000 iterations


In [9]:
print("Fitting 9-mode MNL…")
print(f"  {N_PERSONS} observations, {N_PARAMS_MNL} parameters")
print(f"  Method: {'L-BFGS-B' if bounds else 'BFGS'}, start from perturbed truth")

result_mnl = minimize(
    mnl_log_likelihood,
    x0=x0,
    method="L-BFGS-B" if bounds else "BFGS",
    bounds=bounds,
    options={"maxiter": 5_000, "gtol": 1e-8},
)

print(f"\nConverged: {result_mnl.success}")
print(f"Message:   {result_mnl.message}")
print(f"Iterations: {result_mnl.nit}")
print(f"Function evals: {result_mnl.nfev}")
print(f"Final LL:  {-result_mnl.fun:.4f}")

Fitting 9-mode MNL…
  5000 observations, 18 parameters
  Method: BFGS, start from perturbed truth

Converged: False
Message:   Desired error not necessarily achieved due to precision loss.
Iterations: 28
Function evals: 779
Final LL:  -5481.2122


In [10]:
# Unpack estimates
beta_time_hat, beta_cost_hat, asc_hat = unpack_params(result_mnl.x)
ll_mnl = -result_mnl.fun
ll_null = null_log_likelihood()
K = N_PARAMS_MNL
N = N_PERSONS

print(f"LL_null (equal share)  = {ll_null:.4f}")
print(f"LL_final (MLE)         = {ll_mnl:.4f}")
print(f"LL_true_params         = {-mnl_log_likelihood(TRUE_PARAMS):.4f}")
print(f"\nLL improvement over null: {ll_mnl - ll_null:+.2f}")
print(f"LL gap (MLE - true):       {ll_mnl - (-mnl_log_likelihood(TRUE_PARAMS)):+.4f}")

LL_null (equal share)  = -9311.5381
LL_final (MLE)         = -5481.2122
LL_true_params         = -5488.6689

LL improvement over null: +3830.33
LL gap (MLE - true):       +7.4566


---
## 5. Standard Errors

Three SE estimators (following lecture L06):
1. **Hessian-based** — $\text{SE} = \sqrt{\text{diag}(H^{-1})}$ from numerical Hessian
2. **Robust (Sandwich)** — $\text{SE} = \sqrt{\text{diag}(H^{-1} B H^{-1})}$ where $B = \sum g_n g_n^T$

Under correct specification, Hessian ≈ Robust. Divergence signals misspecification.

In [11]:
def compute_hessian(params, fn, eps=1e-5):
    """Numerical Hessian via central finite differences of the gradient."""
    k = len(params)
    H = np.zeros((k, k))
    for i in range(k):
        def grad_i(p):
            return approx_fprime(p, fn, eps)[i]
        H[i] = approx_fprime(params, grad_i, eps)
    return H


def invert_hessian_robust(H, ridge=1e-6):
    """Invert Hessian with ridge regularization for near-singular cases."""
    try:
        H_inv = np.linalg.inv(H)
    except np.linalg.LinAlgError:
        # Add ridge and try again
        H_reg = H + ridge * np.eye(len(H))
        try:
            H_inv = np.linalg.inv(H_reg)
        except np.linalg.LinAlgError:
            H_inv = np.linalg.pinv(H)
    return H_inv


def mnl_scores(params):
    """Compute per-observation scores for sandwich SE estimator.
    
    Analytical score for MNL with mode-specific β_time and availability:
      s_n(β_time_m) = t_mn * (1[chosen==m] - P_n(m))
      s_n(β_cost)   = c_in - Σ_j P_n(j) * c_jn
      s_n(ASC_m)    = 1[chosen==m] - P_n(m)
    """
    beta_time, beta_cost, asc = unpack_params(params)
    
    V = np.zeros((N_PERSONS, N_MODES))
    for j, m in enumerate(MODE_LABELS):
        V[:, j] = asc[m] + beta_time[m] * T[:, j] + beta_cost * C[:, j]
    V[~AVAIL] = -np.inf
    V_max = V.max(axis=1, keepdims=True)
    expV = np.exp(V - V_max)
    denom = expV.sum(axis=1, keepdims=True)
    P = expV / denom  # (N × 9)
    
    chosen_ind = np.zeros((N_PERSONS, N_MODES))
    chosen_ind[np.arange(N_PERSONS), CHOICE] = 1.0
    
    scores = np.zeros((N_PERSONS, N_PARAMS_MNL))
    
    # β_time scores: cols 0-8
    for j in range(N_MODES):
        avail = AVAIL[:, j]
        t_col = np.nan_to_num(T[:, j], nan=0.0)
        scores[:, j] = t_col * (chosen_ind[:, j] - P[:, j]) * avail
    
    # β_cost score: col 9
    cost_chosen = C[np.arange(N_PERSONS), CHOICE]
    cost_expected = np.sum(P * np.nan_to_num(C, nan=0.0), axis=1)
    scores[:, 9] = cost_chosen - cost_expected
    
    # ASC scores: cols 10-17
    for k, m in enumerate(ASC_MODES):
        j = MODE_LABELS.index(m)
        avail = AVAIL[:, j]
        scores[:, 10 + k] = (chosen_ind[:, j] - P[:, j]) * avail
    
    return scores


def robust_se(params):
    """Sandwich (Huber-White) robust standard errors using analytical scores."""
    scores = mnl_scores(params)
    B = scores.T @ scores
    H = compute_hessian(params, mnl_log_likelihood)
    H_inv = invert_hessian_robust(H)
    V_robust = H_inv @ B @ H_inv
    return np.sqrt(np.maximum(np.diag(V_robust), 0))


# Build param_labels early for diagnostics
_param_labels = []
for m in MODE_LABELS:
    _param_labels.append(f"β_time_{m}")
_param_labels.append("β_cost")
for m in ASC_MODES:
    _param_labels.append(f"ASC_{m}")

print("Computing Hessian-based SE…")
H_mnl = compute_hessian(result_mnl.x, mnl_log_likelihood)
H_inv = invert_hessian_robust(H_mnl)
se_hessian = np.sqrt(np.maximum(np.diag(H_inv), 0))

# Flag potentially weakly identified parameters
median_se = np.median(se_hessian[se_hessian > 1e-8])
for i in range(N_PARAMS_MNL):
    if se_hessian[i] > 10 * median_se:
        print(f"  ⚠ Large SE: {_param_labels[i]} — SE={se_hessian[i]:.2e} — may be weakly identified")

print("\nComputing robust (sandwich) SE via analytical scores…")
se_robust = robust_se(result_mnl.x)

print("Done.")

Computing Hessian-based SE…
  ⚠ Large SE: ASC_4wrh — SE=1.22e+00 — may be weakly identified
  ⚠ Large SE: ASC_2wrh — SE=1.48e+00 — may be weakly identified
  ⚠ Large SE: ASC_mrt — SE=8.05e-01 — may be weakly identified

Computing robust (sandwich) SE via analytical scores…
Done.


---
## 6. Parameter Recovery Results

**Recovery criterion**: $|\hat{\theta} - \theta_{true}| < 2 \cdot SE(\hat{\theta})$ for all 18 parameters.

In [12]:
# Build parameter labels
param_labels = []
for m in MODE_LABELS:
    param_labels.append(f"β_time_{m}")
param_labels.append("β_cost")
for m in ASC_MODES:
    param_labels.append(f"ASC_{m}")

# Parameter table
print(f"{'='*85}")
print(f"MNL ESTIMATION RESULTS — 18 Parameters at μ={GUMBEL_SCALE}, {N} Observations")
print(f"{'='*85}")
print(f"{'Parameter':<18} {'True':>8} {'Estimate':>10} {'SE(Hess)':>10} {'SE(Rob)':>10} {'t(Hess)':>8} {'|Δ|<2SE?':>10}")
print(f"{'-'*85}")

recovery_ok = []
for i, (label, truth, est, se_h, se_r) in enumerate(
    zip(param_labels, TRUE_PARAMS, result_mnl.x, se_hessian, se_robust)):
    t_stat = (est - truth) / se_h if se_h > 1e-12 else 0
    within_2se = abs(est - truth) < 2 * se_h
    recovery_ok.append(within_2se)
    flag = "✅" if within_2se else "❌"
    print(f"{label:<18} {truth:>8.4f} {est:>10.4f} {se_h:>10.4f} {se_r:>10.4f} {t_stat:>8.2f} {flag:>10}")

n_recovered = sum(recovery_ok)
print(f"\nRecovery: {n_recovered}/{N_PARAMS_MNL} parameters within 2 SE")
if n_recovered == N_PARAMS_MNL:
    print("✅ ALL PARAMETERS RECOVERED — estimator implementation validated.")
else:
    failed = [param_labels[i] for i, ok in enumerate(recovery_ok) if not ok]
    print(f"⚠ {len(failed)} parameters outside 2 SE: {failed}")
    print(f"  Note: β at μ={GUMBEL_SCALE} scale. VOT is scale-invariant — see §7 of output.")

MNL ESTIMATION RESULTS — 18 Parameters at μ=25.0, 5000 Observations
Parameter              True   Estimate   SE(Hess)    SE(Rob)  t(Hess)   |Δ|<2SE?
-------------------------------------------------------------------------------------
β_time_car          -0.0240    -0.0326     0.0105     0.0101    -0.82          ✅
β_time_moto         -0.0936    -0.0972     0.0041     0.0042    -0.87          ✅
β_time_4wrh         -0.1396    -0.1341     0.0394     0.0417     0.14          ✅
β_time_2wrh         -0.2040    -0.1767     0.0515     0.0367     0.53          ✅
β_time_krl          -0.1088    -0.1115     0.0049     0.0049    -0.56          ✅
β_time_tj           -0.0428    -0.0450     0.0029     0.0029    -0.75          ✅
β_time_royal        -0.0520    -0.0541     0.0043     0.0042    -0.48          ✅
β_time_lrt          -0.0948    -0.0879     0.0124     0.0118     0.55          ✅
β_time_mrt          -0.1192    -0.2122     0.0546     0.0567    -1.70          ✅
β_cost              -0.0568    -0.05

In [13]:
# Goodness-of-fit
lr_stat = -2 * (ll_null - ll_mnl)
rho2 = 1 - ll_mnl / ll_null
rho2_adj = 1 - (ll_mnl - K) / ll_null
aic = -2 * ll_mnl + 2 * K
bic = -2 * ll_mnl + K * np.log(N)

print(f"{'='*55}")
print(f"GOODNESS-OF-FIT — 9-mode MNL")
print(f"{'='*55}")
print(f"  Observations (N)       = {N:>8,}")
print(f"  Parameters (K)         = {K:>8}")
print(f"  LL_null (equal share)  = {ll_null:>12.4f}")
print(f"  LL_final (MLE)         = {ll_mnl:>12.4f}")
print(f"  LL_true_params         = {-mnl_log_likelihood(TRUE_PARAMS):>12.4f}")
print(f"  LR statistic           = {lr_stat:>12.4f}  (χ²({K}), p ≈ 0)")
print(f"  McFadden ρ²            = {rho2:>12.4f}")
print(f"  Adjusted ρ²            = {rho2_adj:>12.4f}")
print(f"  AIC                    = {aic:>12.2f}")
print(f"  BIC                    = {bic:>12.2f}")
print(f"\n  ρ² = {rho2:.4f}: model captures {rho2*100:.1f}% of information vs null.")
print(f"  LL(MLE) vs LL(true): {ll_mnl - (-mnl_log_likelihood(TRUE_PARAMS)):+.4f} — MLE slightly beats truth (sampling noise).")

GOODNESS-OF-FIT — 9-mode MNL
  Observations (N)       =    5,000
  Parameters (K)         =       18
  LL_null (equal share)  =   -9311.5381
  LL_final (MLE)         =   -5481.2122
  LL_true_params         =   -5488.6689
  LR statistic           =    7660.6517  (χ²(18), p ≈ 0)
  McFadden ρ²            =       0.4114
  Adjusted ρ²            =       0.4094
  AIC                    =     10998.42
  BIC                    =     11115.73

  ρ² = 0.4114: model captures 41.1% of information vs null.
  LL(MLE) vs LL(true): +7.4566 — MLE slightly beats truth (sampling noise).


---
## 7. Implied Value of Travel Time (VOT)

Recovered VOT from estimated parameters:

$$\text{VOT}_m = \frac{\beta_{time,m}}{\beta_{cost}} \times 60,000 \quad [\text{Rp/hour}]$$

In [14]:
# VOT recovery: β_time/β_cost is scale-invariant
# Compare VOT (not raw β_time which is at μ=25 scale)
ILAHI_VTTS = {
    "car": 25200, "moto": 98840, "4wrh": 147280, "2wrh": 215490,
    "krl": 114930, "tj": 45220, "royal": 55000, "lrt": 100000, "mrt": 126000,
}

print(f"{'Mode':<8} {'VOT(true)':>12} {'VOT(est)':>12} {'Err %':>8}  {'βt est(μ=25)':>14}")
print(f"{'-'*65}")
for m in MODE_LABELS:
    vot_true = ILAHI_VTTS[m]
    vot_est = (beta_time_hat[m] / beta_cost_hat) * 60_000
    err_pct = (vot_est / vot_true - 1) * 100
    flag = " ⚠" if abs(err_pct) > 15 else " ✅"
    print(f"{m:<8} {vot_true:>12,.0f} {vot_est:>12,.0f} {err_pct:>+7.1f}%{flag} {beta_time_hat[m]:>14.6f}")

print(f"\nβ_cost Ilahi = {TRUE_DGP['beta_cost']:.4f}, β_cost estimated = {beta_cost_hat:.6f} (at μ={GUMBEL_SCALE})")
print(f"β_cost ratio: true_scaled = {TRUE_DGP_SCALED['beta_cost']:.6f}")
print(f"VOT = β_time/β_cost × 60,000 — scale-invariant.")

Mode        VOT(true)     VOT(est)    Err %    βt est(μ=25)
-----------------------------------------------------------------
car            25,200       38,251   +51.8% ⚠      -0.032614
moto           98,840      113,971   +15.3% ⚠      -0.097173
4wrh          147,280      157,272    +6.8% ✅      -0.134092
2wrh          215,490      207,287    -3.8% ✅      -0.176736
krl           114,930      130,816   +13.8% ✅      -0.111536
tj             45,220       52,726   +16.6% ⚠      -0.044955
royal          55,000       63,406   +15.3% ⚠      -0.054061
lrt           100,000      103,125    +3.1% ✅      -0.087926
mrt           126,000      248,840   +97.5% ⚠      -0.212165

β_cost Ilahi = -1.4200, β_cost estimated = -0.051157 (at μ=25.0)
β_cost ratio: true_scaled = -0.056800
VOT = β_time/β_cost × 60,000 — scale-invariant.


---
## 8. IIA Violation — Red Bus / Blue Bus Test

The IIA property is the MNL's Achilles' heel. We demonstrate by:
1. Computing baseline choice probabilities from the estimated model
2. Adding a "fake KRL Express" (identical to KRL but +1 ASC)
3. Showing MNL predicts transit share doubles (instead of ~constant)

In [15]:
# Use a representative person from zone J2 (all modes available)
# Pick a middle-income person with both car and moto ownership
j2_persons = df_persons[(df_persons["zone_id"] == "J2") & (df_persons["income_segment"] == "mid")]
rep = j2_persons.iloc[0]

print(f"Representative person: zone={rep['zone_id']}, income={rep['income_segment']}, "
      f"car={rep['car_owner']}, moto={rep['moto_owner']}")

# Compute baseline probabilities
V_rep = np.array([asc_hat[m] + beta_time_hat[m] * rep[f"time_{m}"] + beta_cost_hat * rep[f"cost_{m}"]
                  for m in MODE_LABELS])
# Check availability
avail_rep = np.array([not np.isnan(rep[f"time_{m}"]) for m in MODE_LABELS])
V_rep[~avail_rep] = -np.inf

V_max = V_rep[avail_rep].max()
expV = np.exp(V_rep - V_max)
P_base = expV / expV.sum()

print("\nBaseline MNL choice probabilities (J2, mid-income):\n")
for m, p in zip(MODE_LABELS, P_base):
    if p > 0.001:
        print(f"  {m:6s}: {p*100:5.1f}%")
    else:
        print(f"  {m:6s}:    —")

# IIA test: add "KRL Express" — identical to KRL but ASC + 0.1 higher
print("\n--- Adding 'KRL Express' (identical to KRL, ASC = ASC_krl + 0.1) ---\n")

V_iia = np.append(V_rep, V_rep[MODE_LABELS.index("krl")] + 0.1)
avail_iia = np.append(avail_rep, True)
V_iia[~avail_iia] = -np.inf

V_max_iia = V_iia[avail_iia].max()
expV_iia = np.exp(V_iia - V_max_iia)
P_iia = expV_iia / expV_iia.sum()

mode_labels_iia = MODE_LABELS + ["KRL_Express"]
print("After adding KRL Express (IIA prediction):\n")
for m, p in zip(mode_labels_iia, P_iia):
    if p > 0.001:
        print(f"  {m:12s}: {p*100:5.1f}%")
    else:
        print(f"  {m:12s}:    —")

p_krl_before = P_base[MODE_LABELS.index("krl")]
p_krl_after = P_iia[MODE_LABELS.index("krl")]
p_krl_express = P_iia[-1]

print(f"\nKRL share: {p_krl_before*100:.1f}% → {p_krl_after*100:.1f}% (+{p_krl_express*100:.1f}% for Express)")
print(f"Transit (KRL + Express) share: {(p_krl_after + p_krl_express)*100:.1f}%")
print(f"\n⚠ IIA violation: MNL predicts transit share nearly doubles because")
print(f"  it treats KRL and KRL_Express as independent alternatives.")
print(f"  In reality, they are close substitutes — the Nested Logit (03 notebook) fixes this.")

Representative person: zone=J2, income=mid, car=0, moto=1

Baseline MNL choice probabilities (J2, mid-income):

  car   :   0.9%
  moto  :  10.7%
  4wrh  :    —
  2wrh  :    —
  krl   :  15.3%
  tj    :  38.9%
  royal :  23.0%
  lrt   :  11.2%
  mrt   :    —

--- Adding 'KRL Express' (identical to KRL, ASC = ASC_krl + 0.1) ---

After adding KRL Express (IIA prediction):

  car         :   0.8%
  moto        :   9.1%
  4wrh        :    —
  2wrh        :    —
  krl         :  13.1%
  tj          :  33.2%
  royal       :  19.7%
  lrt         :   9.6%
  mrt         :    —
  KRL_Express :  14.5%

KRL share: 15.3% → 13.1% (+14.5% for Express)
Transit (KRL + Express) share: 27.6%

⚠ IIA violation: MNL predicts transit share nearly doubles because
  it treats KRL and KRL_Express as independent alternatives.
  In reality, they are close substitutes — the Nested Logit (03 notebook) fixes this.


---
## 9. SE Divergence as Misspecification Check

Under correct specification, Hessian SE ≈ Robust SE. If they diverge by > 5%, the model
may be misspecified. This previews the Nested Logit (notebook 03): the SE divergence
will be larger when we fit MNL to data generated from a NL DGP.

In [16]:
print(f"{'Parameter':<18} {'SE(Hess)':>10} {'SE(Rob)':>10} {'Dispersion':>10}")
print(f"{'-'*50}")
max_dispersion = 0
for i, label in enumerate(param_labels):
    dispersion = abs(se_robust[i] / se_hessian[i] - 1) * 100 if se_hessian[i] > 1e-12 else 0
    max_dispersion = max(max_dispersion, dispersion)
    flag = " ⚠" if dispersion > 5 else ""
    print(f"{label:<18} {se_hessian[i]:>10.4f} {se_robust[i]:>10.4f} {dispersion:>9.1f}%{flag}")

print(f"\nMax SE dispersion: {max_dispersion:.1f}%")
if max_dispersion < 5:
    print("✅ SE estimators agree — correctly specified MNL on MNL data.")
else:
    print(f"⚠ SE dispersion > 5% — possible misspecification. NL in 03 may be needed.")

Parameter            SE(Hess)    SE(Rob) Dispersion
--------------------------------------------------
β_time_car             0.0105     0.0101       4.0%
β_time_moto            0.0041     0.0042       2.2%
β_time_4wrh            0.0394     0.0417       5.8% ⚠
β_time_2wrh            0.0515     0.0367      28.7% ⚠
β_time_krl             0.0049     0.0049       0.4%
β_time_tj              0.0029     0.0029       0.4%
β_time_royal           0.0043     0.0042       3.7%
β_time_lrt             0.0124     0.0118       5.1% ⚠
β_time_mrt             0.0546     0.0567       3.7%
β_cost                 0.0079     0.0077       2.9%
ASC_car                0.3442     0.3494       1.5%
ASC_moto               0.1086     0.1081       0.4%
ASC_4wrh               1.2221     1.3371       9.4% ⚠
ASC_2wrh               1.4789     1.1295      23.6% ⚠
ASC_tj                 0.1583     0.1591       0.5%
ASC_royal              0.2467     0.2452       0.6%
ASC_lrt                0.5023     0.4807       4.3%
ASC

---
## 10. Parameter Gradient Check

Verify the analytical gradient against numerical gradient at the MLE solution.
At the MLE, the gradient should be ≈ 0 (first-order condition).

In [17]:
grad_at_mle = approx_fprime(result_mnl.x, mnl_log_likelihood, 1e-6)
grad_norm = np.linalg.norm(grad_at_mle)
max_grad = np.max(np.abs(grad_at_mle))

print(f"Gradient at MLE:")
print(f"  ||∇LL|| = {grad_norm:.2e}")
print(f"  max|∇LL| = {max_grad:.2e}")
if max_grad < 1e-3:
    print(f"  ✅ Gradient near zero — first-order condition satisfied.")
else:
    print(f"  ⚠ Gradient not zero — check convergence.")

print(f"\nTop 5 largest gradient components:")
top_idx = np.argsort(np.abs(grad_at_mle))[-5:][::-1]
for idx in top_idx:
    print(f"  {param_labels[idx]:<18}: {grad_at_mle[idx]:+.2e}")

Gradient at MLE:
  ||∇LL|| = 2.04e+00
  max|∇LL| = 1.55e+00
  ⚠ Gradient not zero — check convergence.

Top 5 largest gradient components:
  β_time_tj         : +1.55e+00
  β_time_moto       : +8.13e-01
  β_cost            : +6.40e-01
  β_time_krl        : +5.79e-01
  β_time_royal      : +5.25e-01


---
## 11. Export and Summary

Export estimated parameters for downstream notebooks (03 NL, 04 policy).

In [18]:
# Export estimated parameters
import json

# Compute VOT per mode (scale-invariant)
vot_estimated = {}
for m in MODE_LABELS:
    vot_estimated[m] = float((beta_time_hat[m] / beta_cost_hat) * 60_000)

export = {
    "gumbel_scale": GUMBEL_SCALE,
    "beta_time": {m: float(beta_time_hat[m]) for m in MODE_LABELS},
    "beta_cost": float(beta_cost_hat),
    "asc": {m: float(asc_hat[m]) for m in MODE_LABELS},
    "vot_rp_hr": vot_estimated,
    "ilahi_vot_rp_hr": ILAHI_VTTS,
    "se_beta_time": {},
    "se_asc": {},
    "se_beta_cost": None,
    "goodness_of_fit": {
        "ll_null": ll_null,
        "ll_final": ll_mnl,
        "rho2": rho2,
        "rho2_adj": rho2_adj,
        "aic": aic,
        "bic": bic,
        "n_obs": int(N),
        "n_params": int(K),
    },
    "recovery": {
        "n_recovered": int(n_recovered),
        "n_total": N_PARAMS_MNL,
        "all_within_2se": bool(n_recovered == N_PARAMS_MNL),
    },
}

# Fill SEs
for i, m in enumerate(MODE_LABELS):
    export["se_beta_time"][m] = float(se_hessian[i])
export["se_beta_cost"] = float(se_hessian[9])
for j, m in enumerate(ASC_MODES):
    export["se_asc"][m] = float(se_hessian[10 + j])
export["se_asc"][REF_MODE] = 0.0  # fixed

with open(DATA_DIR / "mnl_estimates.json", "w") as f:
    json.dump(export, f, indent=2)

print(f"✅ Exported: {DATA_DIR / 'mnl_estimates.json'}")
print(f"   {len(json.dumps(export)):,} bytes")
print(f"   Includes: β at μ={GUMBEL_SCALE} scale + VOT in Rp/hr (scale-invariant)")

✅ Exported: /Users/Dhanes/Documents/portfolio/jabodetabek-transity-equity-mapper/notebooks/trans-eng-final/data/mnl_estimates.json
   1,865 bytes
   Includes: β at μ=25.0 scale + VOT in Rp/hr (scale-invariant)


In [19]:
# Final summary
print(f"{'='*65}")
print(f"02 MNL ESTIMATION — COMPLETE")
print(f"{'='*65}")
print(f"  Data:          {N:,} persons, {N_MODES} modes")
print(f"  Parameters:    {K} (9 β_time + 1 β_cost + 8 ASCs)")
print(f"  Gumbel scale:  μ = {GUMBEL_SCALE} (VOT scale-invariant)")
print(f"  Converged:     {result_mnl.success}")
print(f"  LL_final:      {ll_mnl:.4f}")
print(f"  ρ²:            {rho2:.4f}")
print(f"  Recovery:      {n_recovered}/{N_PARAMS_MNL} within 2 SE (at μ={GUMBEL_SCALE})")
print(f"  SE dispersion: {max_dispersion:.1f}%")
print(f"  Exports:       mnl_estimates.json")
print(f"\nNext: 03_nl_estimation.ipynb — Nested Logit with 3 nests")

02 MNL ESTIMATION — COMPLETE
  Data:          5,000 persons, 9 modes
  Parameters:    18 (9 β_time + 1 β_cost + 8 ASCs)
  Gumbel scale:  μ = 25.0 (VOT scale-invariant)
  Converged:     False
  LL_final:      -5481.2122
  ρ²:            0.4114
  Recovery:      18/18 within 2 SE (at μ=25.0)
  SE dispersion: 28.7%
  Exports:       mnl_estimates.json

Next: 03_nl_estimation.ipynb — Nested Logit with 3 nests


---
## Next: `03_nl_estimation.ipynb`

The Nested Logit notebook will:
1. Read `data/persons_jkt.csv` + `data/mnl_estimates.json`
2. Generate new synthetic choices from the NL DGP (with nest correlation)
3. Estimate 3-nest NL parameters via scipy MLE (nest = Own Vehicle, Ridehailing, Transit)
4. Show NL fits better than MNL via LR test
5. Show SE dispersion as misspecification diagnostic (MNL on NL data → large dispersion)